In [ ]:
"""
Module: Loan Risk Model Training Pipeline
Author: Lincoln
Description: End-to-end training pipeline including data cleaning, financial 
             feature engineering, and hyperparameter optimization using PySpark ML.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# Initialize production-grade Spark Session
spark = SparkSession.builder \
    .appName("Loan-Risk-Training-Pipeline") \
    .getOrCreate()

# 1. Data Ingestion & Missing Value Imputation
training_data_path = r"C:\Users\Lincoln\loan prediction dataset\train_u6lujuX_CVtuZ9i.csv"
raw_df = spark.read.csv(training_data_path, header=True, inferSchema=True)

cleaned_df = raw_df.na.fill({
    'LoanAmount': 146, 
    'Loan_Amount_Term': 360, 
    'Credit_History': 1
})

categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed']
cleaned_df = cleaned_df.na.fill("Unknown", subset=categorical_cols)

# 2. Advanced Feature Engineering (Debt-to-Income Ratio)
# Normalizing LoanAmount to exact dollar format and applying division safeguard.
engineered_df = cleaned_df.withColumn(
    "DebtToIncomeRatio", 
    (col("LoanAmount") * 1000) / (col("ApplicantIncome") + col("CoapplicantIncome") + 1.0)
)

# 3. ML Pipeline Component Definition
label_indexer = StringIndexer(inputCol="Loan_Status", outputCol="booking_status_index")

text_features = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Dependents']
pipeline_stages = [label_indexer]

for feature_name in text_features:
    string_indexer = StringIndexer(inputCol=feature_name, outputCol=feature_name + '_index', handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=string_indexer.getOutputCol(), outputCol=feature_name + '_vec', dropLast=False)
    pipeline_stages += [string_indexer, encoder]

numerical_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'DebtToIncomeRatio']
vector_inputs = [feature_name + '_vec' for feature_name in text_features] + numerical_features

assembler = VectorAssembler(inputCols=vector_inputs, outputCol='features')
pipeline_stages.append(assembler)

# 4. Train/Test Data Segmentation
train_data, test_data = engineered_df.randomSplit([0.8, 0.2], seed=42)

# 5. Model Architecture & Hyperparameter Optimization Tuning
dtc_estimator = DecisionTreeClassifier(
    featuresCol='features', 
    labelCol='booking_status_index', 
    predictionCol='prediction'
)

tuning_stages = pipeline_stages + [dtc_estimator]
tuning_pipeline = Pipeline(stages=tuning_stages)

param_grid = ParamGridBuilder() \
    .addGrid(dtc_estimator.maxDepth, [2, 5, 10, 20]) \
    .addGrid(dtc_estimator.maxBins, [10, 20, 40]) \
    .addGrid(dtc_estimator.minInstancesPerNode, [1, 2, 4]) \
    .build()

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol='booking_status_index', 
    predictionCol='prediction', 
    metricName='accuracy'
)

tvs = TrainValidationSplit(
    estimator=tuning_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=accuracy_evaluator,
    trainRatio=0.8,
    seed=42
)

# Execute optimization routine
tvs_model = tvs.fit(train_data)
best_model = tvs_model.bestModel

# 6. Model Exportation and Artifact Storage
model_save_path = r"C:\Users\Lincoln\loan prediction dataset\production_loan_model"
best_model.write().overwrite().save(model_save_path)
print("Pipeline optimization complete. Model artifact saved successfully.")